# Ditaso (2026) - Data processing: factors

This notebook mines and processes the factor data needed. Read the [README.md](README.md) file for context about the project.

## About the data and methodology

**Sources:**

- Fama-French 6 factors from [sakethaleti.com/data](https://www.sakethaleti.com/data), or [this Google Drive link](https://drive.google.com/file/d/1TJqtc-8KlwTmtS6V1rGn8xCj2I0mUdo_). Already at the 1-minute level.

**Filters:**

- Sample period (same as stocks):
    - Range: 2001-01-02 to 2017-12-29. Stocks that are listed or delisted during this period are included.
    - Hours: regular trading hours (09:30 - 16:00 ET).
    - Days: regular trading days, removing days with market-wide halts and diminished trading hours.

**Quality checks:**

- Data is very trusted. There should be data for all and only moments the market is open. Specially: there shouldn't be data for a stock and not for the factors; and there should be all 390 minutes of a day with data.
- Assert that the data is in returns (not log).
- Plot the number of observations per day across time, listing the days with less than 390.

**Processing steps:**

- Download the data into `"data/ff6_1min_returns_raw.csv"`. Alternatively, could be read directly from Google Drive.
- Run the quality checks above.
- Apply the filters above.
- Transform data into log-returns.
- Aggregate the data into 5-minute blocks (09:35, 09:40, ..., 16:00) by summing the returns of the label minute and the 4 minutes before it. Save it into `"data/ff6_5min_returns.csv"`.

## Code setup

In [1]:
%reset -f
import importlib

In [4]:
import os
if os.path.basename(os.getcwd()) == "src": os.chdir("..")

import polars as pl
import numpy as np

import pandas_market_calendars as mcal

import plotnine as gg
from plotnine import ggplot, aes

import src.utils as ut
from src.parameters import PARAMETERS as PARS
importlib.reload(ut)

<module 'src.utils' from 'c:\\Users\\ricar\\OneDrive\\A-Trabalho\\Freelance\\ra-ditaso2026\\src\\utils.py'>

## Importing and dates

Importing and formatting dates as datetime. First look at the data:

In [ ]:
data = (
    pl.read_csv("data/ff6_1min_returns_raw.csv")
    .with_columns(
        pl.col("datetime")
        .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", )
    )
)
cols_fct = ["ff__mkt", "ff__smb", "ff__hml", "ff__rmw", "ff__cma", "ff__umd"]

print(f"Data shape: {data.shape}")
print("First rows:")
data.head(3)

Data shape: (2454420, 7)
First rows:


datetime,ff__mkt,ff__smb,ff__hml,ff__rmw,ff__cma,ff__umd
datetime[μs],f64,f64,f64,f64,f64,f64
1996-01-02 09:30:00,-0.002509,0.001133,-0.000592,-0.000541,-0.000922,-0.003535
1996-01-02 09:31:00,-0.000061,0.000184,0.000053,0.00002,-0.000021,0.000204
1996-01-02 09:32:00,-0.000123,0.000121,-0.000045,0.000123,-0.000098,-0.000083


Checking sorting:

In [27]:
if not data["datetime"].is_sorted():
    raise ValueError("Data is not sorted by datetime.")

All missing dates are non-trading days in NYSE's calendar (assumed as the same as NASDAQ and AMEX). 

In [40]:
date_days = data["datetime"].dt.date().unique()
date_gaps = date_days.diff(1, "drop").dt.total_days()

date_miss = []
for day, gap in zip(
    date_days[1:].filter(date_gaps > 1),
    date_gaps.filter(date_gaps > 1)
):
    date_miss.extend(np.arange(day - np.timedelta64(gap - 1, "D"), day).astype("datetime64[D]"))
date_miss = np.array(date_miss, dtype = "datetime64[D]")

date_valid = (
    mcal.get_calendar("NYSE") # Same as NASDAQ and AMEX
    .valid_days(PARS.date_start, PARS.date_end)
    .to_numpy("datetime64[D]")
)

date_miss_valid = date_miss[np.isin(date_miss, np.array(date_valid))]

if date_miss_valid.size > 0:
    raise ValueError(
        f"There are {len(date_miss_valid)} gaps in days. They begin at:" +
        ut.format_days_byyear(date_miss_valid)
    )

Checking non-full days (less than 391 minutes of data). All are special days given by the NYSE's calendar. Will be removed.

In [54]:
dates_non_full = PARS.trading_days[
    ~ np.isin(PARS.trading_days, PARS.trading_days_full)
]
print(f"Non-full trading days:" + ut.format_days_byyear(dates_non_full))

data = data.filter(np.isin(data["datetime"].dt.date(), PARS.trading_days_full))

Non-full trading days:
- 2001: 07-03, 11-23, 12-24
- 2002: 07-05, 11-29, 12-24
- 2003: 07-03, 11-28, 12-24, 12-26
- 2004: 11-26
- 2005: 11-25
- 2006: 07-03, 11-24
- 2007: 12-24
- 2015: 11-27, 12-24
- 2016: 11-25
- 2017: 07-03, 11-24


## Log-returns and 5-minute aggregation

Data is in returns, and indeed all values are in the range [-1, 1]. We can transform it into log-returns by applying the map $r \to log(1 + r)$.

In [57]:
data_1min = data.with_columns(
    pl.col(cols_fct).log1p()
)

data_1min

datetime,ff__mkt,ff__smb,ff__hml,ff__rmw,ff__cma,ff__umd
datetime[μs],f64,f64,f64,f64,f64,f64
2001-01-02 09:30:00,-0.004918,0.00636,-0.001458,-0.002883,-0.000977,-0.011494
2001-01-02 09:31:00,0.000095,-0.00038,-0.000202,0.000268,0.000314,-0.000224
2001-01-02 09:32:00,-0.000312,-0.00004,0.00021,0.000598,0.000524,0.000227
2001-01-02 09:33:00,-0.000307,-0.000065,0.000177,0.000557,0.000068,-0.000511
2001-01-02 09:34:00,-0.001305,0.000494,0.000684,0.000274,0.000593,0.000511
…,…,…,…,…,…,…
2017-12-29 15:56:00,-0.00003,0.00007,-0.000064,0.000003,-0.00009,0.000055
2017-12-29 15:57:00,-0.000033,0.000039,-0.00004,0.000023,-0.000023,0.000103
2017-12-29 15:58:00,-0.000192,-0.000075,-0.000107,0.000056,0.000017,0.000075


Aggregating below. It is important that the minute blocks are (9:30, 9:35, ...).

In [58]:
data_5min = (
    data_1min.group_by_dynamic("datetime", every="5m")
    .agg([pl.col(col).sum() for col in cols_fct])
)

data_5min

datetime,ff__mkt,ff__smb,ff__hml,ff__rmw,ff__cma,ff__umd
datetime[μs],f64,f64,f64,f64,f64,f64
2001-01-02 09:30:00,-0.006746,0.006369,-0.000591,-0.001186,0.000522,-0.011491
2001-01-02 09:35:00,-0.000719,-0.001081,0.000788,0.002525,0.000066,-0.000575
2001-01-02 09:40:00,0.00256,-0.002862,-0.002178,0.000718,-0.000589,-0.002095
2001-01-02 09:45:00,-0.000581,-0.000438,-0.000018,-0.000494,0.00174,-0.000943
2001-01-02 09:50:00,-0.000909,-0.000451,0.000601,0.000339,0.002158,-0.000354
…,…,…,…,…,…,…
2017-12-29 15:40:00,-0.000458,-0.000037,-0.00016,0.000045,0.000087,0.000097
2017-12-29 15:45:00,0.000247,-0.000531,0.000275,-0.000324,0.000167,-0.000144
2017-12-29 15:50:00,-0.001065,-0.000379,0.000221,-0.000101,0.000517,-0.000404


## Exporting

Saving as .csv:

In [59]:
data_1min.write_csv("data/ff6_1min_returns.csv")
data_5min.write_csv("data/ff6_5min_returns.csv")